# Olist 巴西电商真实数据分析

- 数据来源：**Olist 巴西电商公开数据集**（Kaggle，2016-09 ~ 2018-10，约 9.9 万订单）
- 数据真实、公开、可匿名下载；本 notebook 与 `scripts/`、`sql/olist/` 配套
- 说明：这是真实订单数据，**不是**浏览/点击行为日志；漏斗口径为"订单状态漏斗"（下单→付款→发货→送达/取消）
- 详细报告见 `report/分析报告_olist.md`

## 0. 环境与连接

In [ ]:
import os, pandas as pd
import pymysql

base = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
def load_env(path):
    d = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or "=" not in line: continue
            k, v = line.split("=", 1); d[k.strip()] = v.strip()
    return d

cfg = load_env("../config/db.env")
conn = pymysql.connect(host=cfg["host"], port=int(cfg["port"]), user=cfg["user"],
                       password=cfg["password"], database=cfg["database"], charset="utf8mb4")
def q(sql): return pd.read_sql(sql, conn)

tables = pd.read_sql("SHOW TABLES", conn)
tables[tables.columns[0]].tolist()

## 1. 数据概览

In [ ]:
overview = pd.DataFrame({
    "表": ["olist_orders","olist_order_items","olist_order_payments","olist_order_reviews",
           "olist_customers","olist_products","olist_sellers"],
    "行数": [99441,112650,103886,99224,99441,32951,3095]
})
overview

## 2. 订单状态漏斗（真实运营漏斗）

In [ ]:
funnel = q("""
SELECT '下单' AS 阶段, COUNT(*) AS 订单数 FROM olist_orders
UNION ALL SELECT '付款批准', COUNT(*) FROM olist_orders WHERE order_approved_at IS NOT NULL
UNION ALL SELECT '已发货', COUNT(*) FROM olist_orders WHERE order_delivered_carrier_date IS NOT NULL
UNION ALL SELECT '已送达', COUNT(*) FROM olist_orders WHERE order_delivered_customer_date IS NOT NULL
UNION ALL SELECT '已取消', COUNT(*) FROM olist_orders WHERE order_status='canceled'
""")
funnel

## 3. GMV、客单价与支付结构

In [ ]:
gmv = q("""
SELECT COUNT(*) AS 订单数, ROUND(SUM(payment_value),2) AS 总GMV_BRL, ROUND(AVG(payment_value),2) AS 客单价_BRL
FROM (SELECT order_id, SUM(payment_value) AS payment_value FROM olist_order_payments GROUP BY order_id) t
""")
gmv

In [ ]:
pay = q("""
SELECT payment_type AS 支付方式, COUNT(*) AS 笔数,
       ROUND(SUM(payment_value)*100.0/SUM(SUM(payment_value)) OVER (),2) AS 金额占比_pct
FROM olist_order_payments GROUP BY payment_type ORDER BY 金额占比_pct DESC
""")
pay

## 4. 时间规律

In [ ]:
monthly = q("""
SELECT DATE_FORMAT(order_purchase_timestamp,'%Y-%m') AS 月份, COUNT(*) AS 订单数
FROM olist_orders GROUP BY 月份 ORDER BY 月份
""")
monthly

## 5. 客户价值分层（R+M，F 单独看）

In [ ]:
fd = q("""
WITH cus AS (
    SELECT c.customer_unique_id AS cid, COUNT(DISTINCT o.order_id) AS f
    FROM olist_orders o JOIN olist_customers c ON o.customer_id=c.customer_id
    GROUP BY c.customer_unique_id
)
SELECT f AS 购买次数, COUNT(*) AS 客户数 FROM cus GROUP BY f ORDER BY f
""")
fd

In [ ]:
rm = q("""
WITH cus AS (
    SELECT c.customer_unique_id AS cid,
           DATEDIFF((SELECT MAX(order_purchase_timestamp) FROM olist_orders), MAX(o.order_purchase_timestamp)) AS R,
           COALESCE(SUM(p.payment_value),0) AS M
    FROM olist_orders o JOIN olist_customers c ON o.customer_id=c.customer_id
    LEFT JOIN olist_order_payments p ON o.order_id=p.order_id
    GROUP BY c.customer_unique_id
), tiered AS (
    SELECT cid, R, M,
           NTILE(3) OVER (ORDER BY M DESC) AS m_tier,
           NTILE(3) OVER (ORDER BY R ASC)  AS r_tier
    FROM cus
)
SELECT CASE r_tier WHEN 1 THEN 'R近' WHEN 2 THEN 'R中' ELSE 'R远' END AS 最近购买,
       CASE m_tier WHEN 1 THEN 'M高' WHEN 2 THEN 'M中' ELSE 'M低' END AS 消费金额,
       COUNT(*) AS 客户数, ROUND(SUM(M),2) AS GMV_BRL,
       ROUND(SUM(M)*100.0/SUM(SUM(M)) OVER (),2) AS GMV占比_pct
FROM tiered GROUP BY r_tier, m_tier ORDER BY r_tier, m_tier
""")
rm

## 6. 复购与留存

In [ ]:
rep = q("""
WITH cus AS (
    SELECT c.customer_unique_id AS cid, COUNT(DISTINCT o.order_id) AS f
    FROM olist_orders o JOIN olist_customers c ON o.customer_id=c.customer_id
    GROUP BY c.customer_unique_id
)
SELECT SUM(f=1) AS 单次购买客户, SUM(f>=2) AS 复购客户,
       ROUND(SUM(f>=2)*100.0/COUNT(*),2) AS 复购率_pct
FROM cus
""")
rep

## 7. 品类与地区 GMV

In [ ]:
cat = q("""
SELECT COALESCE(t.product_category_name_english, p.product_category_name) AS 类目,
       ROUND(SUM(i.price+i.freight_value),2) AS GMV_BRL
FROM olist_order_items i
LEFT JOIN olist_products p ON i.product_id=p.product_id
LEFT JOIN olist_product_category_translation t ON p.product_category_name=t.product_category_name
GROUP BY COALESCE(t.product_category_name_english, p.product_category_name)
ORDER BY GMV_BRL DESC LIMIT 10
""")
cat

## 8. 评价与物流

In [ ]:
rev = q("""
SELECT review_score AS 评分, COUNT(*) AS 条数 FROM olist_order_reviews GROUP BY review_score ORDER BY review_score
""")
rev

In [ ]:
dlv = q("""
SELECT COUNT(*) AS 送达订单,
       ROUND(AVG(DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)),1) AS 平均送达天数,
       ROUND(AVG(DATEDIFF(order_estimated_delivery_date, order_purchase_timestamp)),1) AS 平均承诺天数,
       ROUND(SUM(order_delivered_customer_date > order_estimated_delivery_date)*100.0/COUNT(*),2) AS 晚到率_pct
FROM olist_orders
WHERE order_status='delivered' AND order_delivered_customer_date IS NOT NULL
""")
dlv

## 9. 图表（由 scripts/olist_analysis.py 生成）

In [ ]:
from IPython.display import Image, display
import os
charts = sorted(os.listdir("../report/charts_olist"))
for c in charts:
    display(Image(filename=os.path.join("../report/charts_olist", c)))

## 10. 核心结论与局限

**结论**
1. 订单漏斗：下单 99,441 → 付款批准 99.8% → 发货 98.2% → 送达 97.0%；取消率 0.63%
2. GMV 约 1,600 万 BRL，客单价 161 BRL；信用卡支付占 78.3%
3. **复购率仅 3.12%**，96.9% 客户只买 1 单 → RFM 的 F 无区分度，改用 R+M 价值分层；M 高(前 1/3)客户贡献 68% GMV
4. 月度留存率长期为个位数 → 平台以一次性购买为主，运营重心应是拉新与客单价而非留存
5. 2017-11 黑五大促为全年峰值；SP 州贡献 37.5% GMV
6. 平均评分 4.09（J 型分布）；平均送达 12.5 天 vs 承诺 24.4 天，晚到率 8.1%

**局限（必须说明）**
- 巴西市场 2016-2018 数据，非中国电商；币种 BRL，类目为葡萄牙语
- 无浏览/加购日志，漏斗是订单状态口径，与模拟数据"浏览→购买"漏斗不可比
- 数据集本身问题：review_id 有重复、product 列名拼写错误、2016 与 2018-09 之后数据稀少